# Lab 04-2 - Transformations, Scaling, and Pipelines (Solutions)

In [ ]:
## Standard Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

# Let's turn off the FutureWarning messages that are generated by some of the sklearn functions we will be using. 
# These warnings are not relevant to our work and can be safely ignored.
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  

The lab's examples will use data scraped from the Johnson County assessor between 2005 and 2008. These data were originally used by a University of Iowa faculty member to evaluate whether newly listed homes had a list price below what they should be valued based upon their attributes, thereby representing a good deal.

In [ ]:
ic = pd.read_csv("../data/IowaCityHomeSales.csv")
# Print dimensions of data + a blank line for readability
print(ic.shape)
print("\n")

# Let's look at the raw data
ic.head()

In [ ]:
# Print info about all 19 variables. Notice that
# - for many variables the number of non-null values is less than the total number of rows n=777 in the dataset. 
# This means that there are missing values in the dataset.
# - which varialbes are integers, floats, and objects (strings)
print(ic.info())

## Training vs. Testing

In order to prevent [information leakage](https://en.wikipedia.org/wiki/Leakage_(machine_learning)) between the training and testing, it is important to split off a subset of data to be used for validation as soon as possible.  All data exploration (visualization, descriptive stats, etc.), preprocessing (rescaling, transformation, dimension reduction, etc.), and modeling (model choice, hyperparameter tuning, etc.) should be done using *only the training data*.

To create separate training and testing data sets, we'll use the `train_test_split()` function from the `model_selection` module of `sklearn`. The code below:

1. Reserves a randomly chosen subset containing 20% of the available data for the test set, sometimes called the validation set
1. Uses the argument `random_state=7` to set randomization seed that ensures the same random split will occur each time this code is run, thereby allowing replicable randomness. i.e. to work with the same validation data set. The choice of seed value `7` is completely arbitrary. If you use a different seed, you'll get a different random training/testing split. 

In [ ]:
from sklearn.model_selection import train_test_split

# Pay close attention to the arguments i.e. the inputs
train, test = train_test_split(ic, test_size=0.2, random_state=7)

# 80% of n=777 is ~621
print(train.shape)
# 20% of n=777 is ~156
print(test.shape)

We'll now create objects `train_X`, `train_y`, `test_X`, `test_y` such that `train_y` and `test_y` contain the outcome variable `'sale.amount'` and `train_X` and `test_X` contain all numeric predictors from the training and testing data (respectively).

In [ ]:
# Pull out the outcome variable sale.amount
train_y = train['sale.amount']

# Pull out only the numerical predictor variables using the .select_dtypes() method. This will return a 
# dataframe with only the numerical variables.
# We will use this to create our train_x and test_x dataframes.
# We then drop the outcome variable sale.amount from these dataframes using the .drop() method. 
# Recall that axis=0 is for rows and axis=1 is for columns.
train_x = train.select_dtypes(include=['number']).drop('sale.amount', axis=1)

# Inspect the outcome variables for training
print(train_y.head(n=2))

In [ ]:
# Inspect the predictor variables for training
print(train_x.head(n=2))

**Question 1**: Repeat the above to create `test_X` and `test_y`. While this may seem like a brain-dead exercise, take this opportunity to really understand each step of the process

In [ ]:
# Q1: 

# Solution code for Q1)
# Exactly the same two steps as above, with `train` swapped for `test`.
test_y = test['sale.amount']
test_x = test.select_dtypes(include=['number']).drop('sale.amount', axis=1)

# Inspect the outcome and predictor variables for testing
print(test_y.head(n=2))
print(test_x.head(n=2))

## Data Preparation

Now that we've got a separate validation set, we're free to explore the training data and make any modeling choices that we deem appropriate. 

### Exploratory Data Analysis (EDA)
A good first step is to perform an exploratory data analysis of the numeric predictors using data visualizations and descriptive statistics. Some code to do this is given below:

In [ ]:
## Descriptive stats using .describe(), then .round()
print(train.describe().round(2))

## Distributions
train.hist(figsize = (15,12))
plt.show()

### 1. Transformations

Nothing looks too out of the ordinary in the histogram, but we might notice that some predictors, notably `'area.lot'` are extremely right-skewed with some large outliers. We might consider applying a normalizing transformation to this predictor. This code demonstrates a Box-Cox transformation using `PowerTransformer()`. You should note that the example uses the `fit()` method to fit the transformation to the provided data, which amounts to determining and storing the parameters of the transformation. Next, the `transform()` method is used to apply these parameters to actually transform the input data.

In [ ]:
## Import PowerTransformer
from sklearn.preprocessing import PowerTransformer

## Box-Cox transformation. Notice the .fit() and .transform() methods. The .fit() method is used to learn 
# the parameters of the transformation from the training data, 
# and the .transform() method is used to apply the transformation to the training data.
bc_trans = PowerTransformer(method = 'box-cox').fit(train[['area.lot']])
bc_trans_arealot = bc_trans.transform(train[['area.lot']])

## Histogram of area.lot after transformation
pd.DataFrame(bc_trans_arealot).hist()
plt.show()

If the histogram shown above was all that we desired, we could have used the `fit_transform()` method, which combines the two steps. However, when applying our final machine learning approach to new data we do it separately with `.fit()` and `.transform()` since we'll want to be able to fit the transformation on the training data, then apply it later to the test data.

**Question 2**: Consider the use of a Box-Cox transformation as part of a machine learning approach. Identify and briefly explain the problem that arises in the following scenario: We apply the transformation using `fit_transform()` to the entire data set before performing the training and testing split. *Hint*: Think about information leakage.

**Answers to Question 2**:

The Box-Cox transformation is not a fixed formula: its exponent $\lambda$ is *estimated from the data*. So if we call `fit_transform()` on the entire data set before splitting, that $\lambda$ is computed using the test observations as well as the training ones. Two things then go wrong:

1. **The test set stops being a stand-in for genuinely unseen data.** Information about the test observations (their spread, their skew) has been baked into a transformation that the model is then trained through, so the two sets are no longer independent of each other.
1. **Our estimate of test error becomes optimistically biased.** We evaluate the model on data that helped shape its own preprocessing, so it looks better than it would on truly new data -- which is precisely what holding out a validation set was supposed to protect us from.

The fix is the pattern demonstrated above: `.fit()` the transformation on the training data only, then `.transform()` the test data using those stored training parameters.

### 2. Scaling

Based upon our initial data exploration, you should also have noticed that our predictors exist on very different scales. Thus, we must rescale them to ensure the variables with larger units do not dominate.

Each of the rescaling approaches discussed in our lecture slides are briefly demonstrated below on a few of the available predictors:

In [ ]:
## A few predictors to show
example_pred_vars = ['area.lot', 'area.living', 'area.garage1']

## Ex 1: Standard Scaling (Z-scores)
# Import necessary function from library:
from sklearn.preprocessing import StandardScaler

# Compute the parameters of the transformation (here the mean and sd needed for Z-scores):
ss_trans = StandardScaler().fit(train[example_pred_vars])

# Apply the transformation to the training data:
ss_trans_out = ss_trans.transform(train[example_pred_vars])

# Visualize: 
pd.DataFrame(ss_trans_out, columns = example_pred_vars).hist(layout = (1,3), figsize = (12,4))
plt.show()

In [ ]:
## Ex 2: Min-Max Scaling
# Import necessary function from library:
from sklearn.preprocessing import MinMaxScaler

# Compute the parameters of the transformation (here smallest and largest value):
rs_trans = MinMaxScaler().fit(train[example_pred_vars])

# Apply the transformation to the training data:
rs_trans_out = rs_trans.transform(train[example_pred_vars])

# Visualize: 
pd.DataFrame(rs_trans_out, columns = example_pred_vars).hist(layout = (1,3), figsize = (12,4))
plt.show()

Two more standardization functions to know about that aren't shown: robust scaling and max absolute scaling

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler

### 3. Pipelines

One of the most attractive features of `sklearn` are **pipelines**, or sequences of data handling steps that can easily be applied to a variety of machine learning tasks such as training, validation, hyperparameter tuning and more. These are especially useful when running the same set of tasks over and over again. 

The example below demonstrates a simple pipeline that applies a Box-Cox transformation followed by min-max rescaling:

In [ ]:
# THIS IS AN IMPORTANT CODE BLOCK, MAKE SURE YOU SPEND TIME TO UNDERSTAND IT SINCE 
# WE'LL BE USING PIPELINES OFTEN
from sklearn.pipeline import Pipeline 

## Set up the two steps of the pipeline: the transformation and the rescaling
my_pipeline = Pipeline([
    ('transformer', PowerTransformer(method = 'yeo-johnson')),
    ('scaler', MinMaxScaler())
])

## .fit() the pipeline to the subset of 3 variables we visualized earlier in the histograms above,
## and then .transform() the training data using the fitted pipeline
fitted_pipe = my_pipeline.fit(train[example_pred_vars])
train_transformed = fitted_pipe.transform(train[example_pred_vars])

## The output of the pipeline is a numpy array with 621 rows and 3 columns
print(type(train_transformed))
print("\n")
print(train_transformed.shape)
print("\n")

## So we convert the transformed data to a DataFrame for visualization
train_transformed = pd.DataFrame(
    # Create a new dataframe 
    data = train_transformed, 
    columns = example_pred_vars
)
train_transformed.hist(layout = (1,3), figsize = (12,4))
plt.show()

While they are not consequential in this simple example, we should notice that the pipeline's steps are named `'transformer`' and `'scaler'`. These are names that we choose and can subsequently use to refer to certain steps within the pipeline as we see fit e.g. `'transformation`' and `'scaling'`

Later we'll add a $k$-nearest neighbors model to this pipeline.

## Application

To conclude this lab, you will apply the workflow from the example to a new dataset on your own. The data is from a database on breast cancer in Wisconsin. The data was obtained from the UC-Irvine machine learning repository, you can read more about these data [at this link](https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+(diagnostic)).

Each observation consists of the mean values for various cell characteristics of a patient. The goal is to predict the patient's breast cancer diagnosis, recorded as "Label" which is binary. Thus is a classification problem:
* `M` for malignant (cancerous)
* `B` for benign (non-cancerous)

In [ ]:
## Read data
wbc = pd.read_csv("../data/wisc_bc.csv")

print(wbc.shape)
print("\n")

wbc.head()

**Question 3**:
    
- **Part A**: Create a 70% training, 30% testing data split using random_state = 5. Consider naming this `train_wbc` and `test_wbc` so you don't overwrite the original `train` and `test` dataframes.
- **Part B**: Create histograms showing the distributions of the 10 numeric predictors, then decide upon whether a data preparation pipeline should include transformations, rescaling, or both.
- **Part C**: Set up a machine learning pipeline that includes the following steps:
    1. A normalizing transformation 
    2. Rescaling
- **Part D**: After you `.fit()` your pipeline, apply it to the training data and convert it to pandas data frame format
- **Part E**: Compare a histogram of the transformed and rescaled variables to the histogram Part B. Was your pipeline successful?

In [ ]:
# Q3.a)

# Solution code for Q3.a)
train_wbc, test_wbc = train_test_split(wbc, test_size=0.3, random_state=5)
print("The training data has " + str(train_wbc.shape[0]) + " rows and " + str(train_wbc.shape[1]) + " columns")
print("The test data has " + str(test_wbc.shape[0]) + " rows and " + str(test_wbc.shape[1]) + " columns")

train_y_wbc = train_wbc['Label']
test_y_wbc = test_wbc['Label']

# Pull out only the numerical predictor variables.
# Since the outcome variable Label is categorical it is dropped by .select_dtypes(include=['number'])
# Be sure to also drop ID since it is not actually a predictor variable
train_x_wbc = train_wbc.select_dtypes(include=['number']).drop(columns='ID')
test_x_wbc = test_wbc.select_dtypes(include=['number']).drop(columns='ID')

# This leaves the 10 numeric predictors
print(train_x_wbc.shape)

In [ ]:
# Q3.b)

# Solution code for Q3.b)
train_x_wbc.hist(figsize = (20,10))
plt.show()

**Solution for Q3.b):** The pipeline should include **both** a transformation and rescaling.

- **Rescaling is needed** because the predictors live on wildly different scales: `Area` runs from roughly 144 to 2,250, while `Smoothness` runs from about 0.053 to 0.145. Without rescaling, `Area` and `Perimeter` would dominate any distance-based calculation simply because their units are bigger.
- **A transformation is needed** because several predictors are clearly right-skewed, most of all `Concavity` (skewness 1.51), `Area` (1.40), and `Compactness` (1.29).

In [ ]:
# Q3.c) 

# Solution code for Q3.c)
# Same two-step structure as my_pipeline above: normalize first, then rescale.
q3_pipeline = Pipeline([
    ('transformer', PowerTransformer(method = 'yeo-johnson')),
    ('scaler', StandardScaler())
])

In [ ]:
# Q3.d)

# Solution code for Q3.d)
q3_fitted_pipe = q3_pipeline.fit(train_x_wbc)
q3_transformed = q3_fitted_pipe.transform(train_x_wbc)

## The output of the pipeline is a numpy array with 398 rows and 10 columns
print(type(q3_transformed))
print("\n")
print(q3_transformed.shape)
print("\n")

## So we convert the transformed data to a DataFrame for visualization
q3_transformed = pd.DataFrame(
    # Create a new dataframe
    data = q3_transformed,
    columns = train_x_wbc.columns
)
q3_transformed.hist(figsize = (20,10))
plt.show()

**Solution for Q3.e):** Yes, the pipeline did what we wanted, on both counts.

Comparing the Part D histograms to the Part B histograms, the long right tails are gone and every variable is now roughly symmetric and centered. 

Note: The `PowerTransformer` fixed the *shape* of each distribution, and the `StandardScaler` fixed its *center and spread*. Neither step on its own would have been enough, which is exactly why we chained them in a pipeline.